# pivot() and pivot_table()

In [1]:
import pandas as pd
import numpy as np

In [2]:
# STACKED OR LONG FORMAT
# We have multiple rows for each subject variable
data = {
    "value": range(12),
    "variable": ["A"] * 3 + ["B"] * 3 + ["C"] * 3 + ["D"] * 3,
    "date": pd.to_datetime(["2020-01-03", "2020-01-04", "2020-01-05"] * 4),
}
df = pd.DataFrame(data)
df

,value,variable,date
0,0,A,2020-01-03
1,1,A,2020-01-04
2,2,A,2020-01-05
3,3,B,2020-01-03
4,4,B,2020-01-04
5,5,B,2020-01-05
6,6,C,2020-01-03
7,7,C,2020-01-04
8,8,C,2020-01-05
9,9,D,2020-01-03


In [3]:
# RECORD OR WIDE FORMAT
# The unique variables are used as columns, and a dates
# index identifies each unique observation.
df.pivot(columns="variable", index="date", values="value")

variable,A,B,C,D
date,,,,
2020-01-03,0,3,6,9
2020-01-04,1,4,7,10
2020-01-05,2,5,8,11


In [4]:
df["value2"] = df["value"] * 2
df

,value,variable,date,value2
0,0,A,2020-01-03,0
1,1,A,2020-01-04,2
2,2,A,2020-01-05,4
3,3,B,2020-01-03,6
4,4,B,2020-01-04,8
5,5,B,2020-01-05,10
6,6,C,2020-01-03,12
7,7,C,2020-01-04,14
8,8,C,2020-01-05,16
9,9,D,2020-01-03,18


In [5]:
# If we omit the values argument and the df has more than one column
# of values, the result will have hierarchical columns where the topmost
# level indicates the corresponding value column.
pivoted = df.pivot(columns="variable", index="date")
pivoted

value           value2            
variable       A  B  C   D      A   B   C   D
date                                         
2020-01-03     0  3  6   9      0   6  12  18
2020-01-04     1  4  7  10      2   8  14  20
2020-01-05     2  5  8  11      4  10  16  22

In [6]:
# Select subsets from the pivoted df
pivoted["value2"]

variable,A,B,C,D
date,,,,
2020-01-03,0,6,12,18
2020-01-04,2,8,14,20
2020-01-05,4,10,16,22


In [7]:
# If the data contains duplicates or for pivoting
# with aggregation: use pivot_table()
import datetime

df = pd.DataFrame(
    {
        "A": ["one", "one", "two", "three"] * 6,
        "B": ["A", "B", "C"] * 8,
        "C": ["foo", "foo", "foo", "bar", "bar", "bar"] * 4,
        "D": np.random.randn(24),
        "E": np.random.randn(24),
        "F": [datetime.datetime(2013, i, 1) for i in range(1, 13)]
        + [datetime.datetime(2013, i, 15) for i in range(1, 13)],
    }
)
df

,A,B,C,D,E,F
0,one,A,foo,0.101080,-0.933904,2013-01-01
1,one,B,foo,0.200624,-0.102355,2013-02-01
2,two,C,foo,0.538897,0.042128,2013-03-01
3,three,A,bar,-0.143278,0.286760,2013-04-01
4,one,B,bar,1.442100,-1.710128,2013-05-01
5,one,C,bar,-0.168732,-0.118872,2013-06-01
6,two,A,foo,-0.236066,-1.129893,2013-07-01
7,three,B,foo,-0.138989,0.118196,2013-08-01
8,one,C,foo,0.682178,-0.370479,2013-09-01
9,one,A,bar,0.310133,-0.947869,2013-10-01


In [8]:
pd.pivot_table(df, values="D", index=["A", "B"], columns="C")

C             bar       foo
A     B                    
one   A  0.089671  0.492524
      B  0.085963 -0.148683
      C  0.586285  0.742065
three A  0.483052       NaN
      B       NaN  0.794490
      C  1.428898       NaN
two   A       NaN -0.104093
      B  1.194945       NaN
      C       NaN  0.767693

In [9]:
pd.pivot_table(df, values=["D", "E"], index=["B"], columns=["A", "C"], aggfunc="sum")

D                                                          E  \
A       one               three                two                 one   
C       bar       foo       bar       foo      bar       foo       bar   
B                                                                        
A  0.179342  0.985048  0.966103       NaN      NaN -0.208186 -0.313266   
B  0.171926 -0.297366       NaN  1.588981  2.38989       NaN -0.847725   
C  1.172571  1.484130  2.857797       NaN      NaN  1.535387 -0.139717   

                                                     
A               three                 two            
C       foo       bar       foo       bar       foo  
B                                                    
A -1.278965  0.693890       NaN       NaN -1.306559  
B -1.796901       NaN -1.115728  1.591092       NaN  
C -0.765703  0.430988       NaN       NaN -0.131880

In [10]:
pd.pivot_table(df, values="E", index=["B", "C"], columns=["A"], aggfunc=["sum", "mean"])

sum                          mean                    
A           one     three       two       one     three       two
B C                                                              
A bar -0.313266  0.693890       NaN -0.156633  0.346945       NaN
  foo -1.278965       NaN -1.306559 -0.639483       NaN -0.653279
B bar -0.847725       NaN  1.591092 -0.423862       NaN  0.795546
  foo -1.796901 -1.115728       NaN -0.898450 -0.557864       NaN
C bar -0.139717  0.430988       NaN -0.069858  0.215494       NaN
  foo -0.765703       NaN -0.131880 -0.382852       NaN -0.065940

## Adding margins

In [11]:
table = df.pivot_table(index=["A", "B"], columns="C", values=["D", "E"], margins=True, aggfunc="std")
table

D                             E                    
C             bar       foo       All       bar       foo       All
A     B                                                            
one   A  0.311780  0.553585  0.434340  1.118977  0.416375  0.743555
      B  1.917867  0.493995  1.151420  1.819055  1.125849  1.265139
      C  1.067756  0.084693  0.624911  0.069315  0.017497  0.185360
three A  0.885764       NaN  0.885764  0.085114       NaN  0.085114
      B       NaN  1.320139  1.320139       NaN  0.956093  0.956093
      C  0.169124       NaN  0.169124  0.744580       NaN  0.744580
two   A       NaN  0.186638  0.186638       NaN  0.674034  0.674034
      B  0.666481       NaN  0.666481  1.085860       NaN  1.085860
      C       NaN  0.323567  0.323567       NaN  0.152831  0.152831
All      0.919879  0.630739  0.779538  0.861143  0.574799  0.789434

In [12]:
table.stack(future_stack=True)

D         E
A     B C                      
one   A bar  0.311780  1.118977
        foo  0.553585  0.416375
        All  0.434340  0.743555
      B bar  1.917867  1.819055
        foo  0.493995  1.125849
        All  1.151420  1.265139
      C bar  1.067756  0.069315
        foo  0.084693  0.017497
        All  0.624911  0.185360
three A bar  0.885764  0.085114
        foo       NaN       NaN
        All  0.885764  0.085114
      B bar       NaN       NaN
        foo  1.320139  0.956093
        All  1.320139  0.956093
      C bar  0.169124  0.744580
        foo       NaN       NaN
        All  0.169124  0.744580
two   A bar       NaN       NaN
        foo  0.186638  0.674034
        All  0.186638  0.674034
      B bar  0.666481  1.085860
        foo       NaN       NaN
        All  0.666481  1.085860
      C bar       NaN       NaN
        foo  0.323567  0.152831
        All  0.323567  0.152831
All     bar  0.919879  0.861143
        foo  0.630739  0.574799
        All  0.779538  0.789434

# stack() and unstack()

In [13]:
# stack() : pivot a level of the column labels, to return a df
#           with an index with a new inner-most level of row labels.
tuples = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

index = pd.MultiIndex.from_arrays(tuples, names=["first", "second"])
df = pd.DataFrame(np.random.randn(8, 2), index=index, columns=["A", "B"])

df2 = df[:4]
df2

A         B
first second                    
bar   one    -0.744018  0.157970
      two     0.523325  0.173958
baz   one     0.314737  2.320413
      two     0.653629 -0.289166

In [14]:
# Compress a level in the DataFrame (here the horizontal level of A and B)
# to produce a Series. The stacked level becomes the new inner most level
# of the index.
stacked = df2.stack(future_stack=True)
stacked

first  second   
bar    one     A   -0.744018
               B    0.157970
       two     A    0.523325
               B    0.173958
baz    one     A    0.314737
               B    2.320413
       two     A    0.653629
               B   -0.289166
dtype: float64

In [15]:
# The inverse of stack() is unstack(); by default usntacks the last level
stacked.unstack()

A         B
first second                    
bar   one    -0.744018  0.157970
      two     0.523325  0.173958
baz   one     0.314737  2.320413
      two     0.653629 -0.289166

In [16]:
stacked.unstack(level=1)

second        one       two
first                      
bar   A -0.744018  0.523325
      B  0.157970  0.173958
baz   A  0.314737  0.653629
      B  2.320413 -0.289166

In [17]:
stacked.unstack(level=0)

first          bar       baz
second                      
one    A -0.744018  0.314737
       B  0.157970  2.320413
two    A  0.523325  0.653629
       B  0.173958 -0.289166

In [18]:
index = pd.MultiIndex.from_product([[2, 1], ["a", "b"]])
df = pd.DataFrame(np.random.randn(4), index=index, columns=["A"])
df

A
2 a  0.523701
  b  0.544547
1 a -1.088398
  b  1.251187

In [19]:
all(df.unstack().stack(future_stack=True) == df.sort_index())

True

## Stack/Unstack Multiple Levels at the same time

In [20]:
columns = pd.MultiIndex.from_tuples(
    [
        ("A", "cat", "long"),
        ("B", "cat", "long"),
        ("A", "dog", "short"),
        ("B", "dog", "short"),
    ],
    names=["exp", "animal", "hair_length"],
)

df = pd.DataFrame(np.random.randn(4, 4), columns=columns)
df

exp,A,B,A,B
animal,cat,cat,dog,dog
hair_length,long,long,short,short
0,-0.414666,-0.501513,1.013853,0.921497
1,0.021578,0.832831,0.126847,-1.485021
2,-0.008395,0.912861,0.239507,0.947826
3,0.840265,-1.342085,1.055539,0.675140


In [21]:
df.stack(level=["animal", "hair_length"], future_stack=True)

exp                          A         B
  animal hair_length                    
0 cat    long        -0.414666 -0.501513
  dog    short        1.013853  0.921497
1 cat    long         0.021578  0.832831
  dog    short        0.126847 -1.485021
2 cat    long        -0.008395  0.912861
  dog    short        0.239507  0.947826
3 cat    long         0.840265 -1.342085
  dog    short        1.055539  0.675140

## Missing Data

In [22]:
columns = pd.MultiIndex.from_tuples(
    [
        ("A", "cat"),
        ("B", "dog"),
        ("B", "cat"),
        ("A", "dog"),
    ],
    names=["exp", "animal"],
)
index = pd.MultiIndex.from_product(
    [("bar", "baz", "foo", "qux"), ("one", "two")],
    names=["first", "second"],
)
df = pd.DataFrame(np.random.randn(8, 4), index=index, columns=columns)
df3 = df.iloc[[0, 1, 4, 7], [1, 2]]
df3

exp                  B          
animal             dog       cat
first second                    
bar   one     0.461406  1.234276
      two     0.678151  1.554943
foo   one     0.456658  1.059877
qux   two    -0.593725 -0.216061

In [23]:
df3.unstack()

exp            B                              
animal       dog                 cat          
second       one       two       one       two
first                                         
bar     0.461406  0.678151  1.234276  1.554943
foo     0.456658       NaN  1.059877       NaN
qux          NaN -0.593725       NaN -0.216061

# melt() and wide_to_long()

In [24]:
cheese = pd.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    }
)
cheese

,first,last,height,weight
0,John,Doe,5.5,130
1,Mary,Bo,6.0,150


In [25]:
# Unpivot a table from wide to long format i.e.
# to multiple rows for specific subject identifiers (first, last)
cheese.melt(id_vars=["first", "last"])

,first,last,variable,value
0,John,Doe,height,5.5
1,Mary,Bo,height,6.0
2,John,Doe,weight,130.0
3,Mary,Bo,weight,150.0


In [26]:
# Specify another name for the variable name
cheese.melt(id_vars=["first", "last"], var_name="quantity")

,first,last,quantity,value
0,John,Doe,height,5.5
1,Mary,Bo,height,6.0
2,John,Doe,weight,130.0
3,Mary,Bo,weight,150.0


In [27]:
index = pd.MultiIndex.from_tuples([("person", "A"), ("person", "B")])
cheese = pd.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    },
    index=index,
)
cheese

first last  height  weight
person A  John  Doe     5.5     130
       B  Mary   Bo     6.0     150

In [28]:
# Normally, the original index is ignored:
cheese.melt(id_vars=["first", "last"])

,first,last,variable,value
0,John,Doe,height,5.5
1,Mary,Bo,height,6.0
2,John,Doe,weight,130.0
3,Mary,Bo,weight,150.0


In [29]:
# If we don't want the original index to be ignored,
# duplicate index values will be created:
cheese.melt(id_vars=["first", "last"], ignore_index=False)

first last variable  value
person A  John  Doe   height    5.5
       B  Mary   Bo   height    6.0
       A  John  Doe   weight  130.0
       B  Mary   Bo   weight  150.0

In [30]:
# wide_to_long() is similar to melt(), with more customization.
dft = pd.DataFrame(
    {
        "A1970": {0: "a", 1: "b", 2: "c"},
        "A1980": {0: "d", 1: "e", 2: "f"},
        "B1970": {0: 2.5, 1: 1.2, 2: 0.7},
        "B1980": {0: 3.2, 1: 1.3, 2: 0.1},
        "X": dict(zip(range(3), np.random.randn(3))),
    }
)
dft["id"] = dft.index
dft

,A1970,A1980,B1970,B1980,X,id
0,a,d,2.5,3.2,0.017999,0
1,b,e,1.2,1.3,1.051840,1
2,c,f,0.7,0.1,-0.542267,2


In [31]:
pd.wide_to_long(dft, ["A", "B"], i="id", j="year")

,,X,A,B
id,year,,,
0,1970,0.017999,a,2.5
1,1970,1.051840,b,1.2
2,1970,-0.542267,c,0.7
0,1980,0.017999,d,3.2
1,1980,1.051840,e,1.3
2,1980,-0.542267,f,0.1


# get_dummies() and from_dummies()

In [32]:
df = pd.DataFrame(
    {
        "key": list("bbacab"),
        "data1": range(6),
    }
)

# Create a new DataFrame with columns the unique values from "key" column.
# The values indicate the presence of each one of a, b and c, per row.
pd.get_dummies(df["key"])

,a,b,c
0,False,True,False
1,False,True,False
2,True,False,False
3,False,False,True
4,True,False,False
5,False,True,False


In [33]:
df["key"].str.get_dummies()

,a,b,c
0,0,1,0
1,0,1,0
2,1,0,0
3,0,0,1
4,1,0,0
5,0,1,0


In [34]:
# Add a prefix to the column names...
dummies = pd.get_dummies(df["key"], prefix="key")
dummies

,key_a,key_b,key_c
0,False,True,False
1,False,True,False
2,True,False,False
3,False,False,True
4,True,False,False
5,False,True,False


In [35]:
# so we can merge the result with original DataFrame
df[["data1"]].join(dummies)

,data1,key_a,key_b,key_c
0,0,False,True,False
1,1,False,True,False
2,2,True,False,False
3,3,False,False,True
4,4,True,False,False
5,5,False,True,False


In [36]:
# Return an array of ten random samples from the standard normal distribution.
values = np.random.randn(10)
print(values)

[-0.08172213 -0.45259681  0.39597919  0.45217071 -0.69727049  0.97524844
 -0.21932111  0.30180195 -0.67680405 -0.44415712]


In [37]:
# Define the discrete bins
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

# Bin the values into discrete bins: (0.0, 0.2]...
# and use get_dummies() to create a new DataFrame with the bins as
# columns and the values indicating into which bin each sample belongs
pd.get_dummies(pd.cut(values, bins))

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,False
1,False,False,False,False,False
2,False,True,False,False,False
3,False,False,True,False,False
4,False,False,False,False,False
5,False,False,False,False,True
6,False,False,False,False,False
7,False,True,False,False,False
8,False,False,False,False,False
9,False,False,False,False,False


In [38]:
df = pd.DataFrame(
    {
        "A": ["a", "b", "a"],
        "B": ["c", "c", "b"],
        "C": [1, 2, 3],
    }
)
# When passing a DataFrame to get_dummies(), numerical columns stay unaltered.
pd.get_dummies(df)

,C,A_a,A_b,B_b,B_c
0,1,True,False,False,True
1,2,False,True,False,True
2,3,True,False,True,False


In [39]:
simple = pd.get_dummies(df, prefix="new_prefix")
simple

,C,new_prefix_a,new_prefix_b,new_prefix_b,new_prefix_c
0,1,True,False,False,True
1,2,False,True,False,True
2,3,True,False,True,False


In [40]:
from_list = pd.get_dummies(df, prefix=["from_A", "from_B"])
from_list

,C,from_A_a,from_A_b,from_B_b,from_B_c
0,1,True,False,False,True
1,2,False,True,False,True
2,3,True,False,True,False


# explode()

In [41]:
# NOTE: The "values" column contains list-like values.
#       Using explode() on this column, transforms each list-like
#       value, to a separate row.
keys = ["panda1", "panda2", "panda3"]
values = [["eats", "shoots"], ["shoots", "leaves"], ["eats", "leaves"]]
df = pd.DataFrame({"keys": keys, "values": values})
df

,keys,values
0,panda1,"[eats, shoots]"
1,panda2,"[shoots, leaves]"
2,panda3,"[eats, leaves]"


In [42]:
# explode() on a column
df["values"].explode()

0      eats
0    shoots
1    shoots
1    leaves
2      eats
2    leaves
Name: values, dtype: object

In [43]:
# explode on a DataFrame
df.explode("values")

,keys,values
0,panda1,eats
0,panda1,shoots
1,panda2,shoots
1,panda2,leaves
2,panda3,eats
2,panda3,leaves


# crosstab()

In [44]:
a = np.array(["foo", "foo", "bar", "bar", "foo", "foo"], dtype=object)
b = np.array(["one", "one", "two", "one", "two", "one"], dtype=object)
c = np.array(["dull", "dull", "shiny", "dull", "dull", "shiny"], dtype=object)

# Produce a crosstabulation of two or more factors:
# Initially, a frequency table of the factors is produced
# ARGUMENTS index:   a Series that provides the unique values to group by in the rows
#           columns: a list of Series providing unique values to group by in the columns
pd.crosstab(a, [b, c], rownames=["a"], colnames=["b", "c"])

b    one        two      
c   dull shiny dull shiny
a                        
bar    1     0    0     1
foo    2     1    1     0

In [45]:
df = pd.DataFrame({"A": [1, 2, 2, 2, 2], "B": [3, 3, 4, 4, 4], "C": [1, 1, np.nan, 1, 1]})
df

,A,B,C
0,1,3,1.0
1,2,3,1.0
2,2,4,NaN
3,2,4,1.0
4,2,4,1.0


In [46]:
# index=df.A    group by the unique values of column A -> the index
# columns=df.B  ""                                   B -> the columns
# Here the result is a frequency table:
pd.crosstab(df.A, df.B)

B,3,4
A,,
1,1,0
2,1,3


## Normalization

In [47]:
# Produce a frequency tables with percentages instead of counts
pd.crosstab(df.A, df.B, normalize=True)

B,3,4
A,,
1,0.2,0.0
2,0.2,0.6


# cut()
Computes groupings for the input values: it bins the values into discret intervals.<br/>
Used to transform continuous variables -> discrete/categorical variables

In [48]:
# An integer bins (here 3) will create three equal-width bins.
ages = np.array([10, 15, 13, 12, 23, 25, 28, 59, 60])
pd.cut(ages, bins=3)

[(9.95, 26.667], (9.95, 26.667], (9.95, 26.667], (9.95, 26.667], (9.95, 26.667], (9.95, 26.667], (26.667, 43.333], (43.333, 60.0], (43.333, 60.0]]
Categories (3, interval[float64, right]): [(9.95, 26.667] < (26.667, 43.333] < (43.333, 60.0]]

In [49]:
# A list of k values, produces k-1 bins: (0,18] (18,35] (35,70]
pd.cut(ages, bins=[0, 18, 35, 70])

[(0, 18], (0, 18], (0, 18], (0, 18], (18, 35], (18, 35], (18, 35], (35, 70], (35, 70]]
Categories (3, interval[int64, right]): [(0, 18] < (18, 35] < (35, 70]]

In [50]:
# Pass an IntervalIndex to bins
pd.cut(ages, bins=pd.IntervalIndex.from_breaks([0, 40, 70]))

[(0, 40], (0, 40], (0, 40], (0, 40], (0, 40], (0, 40], (0, 40], (40, 70], (40, 70]]
Categories (2, interval[int64, right]): [(0, 40] < (40, 70]]

# factorize()

In [51]:
x = pd.Series(["A", "A", np.nan, "B", 3.14, np.inf])
x

0       A
1       A
2     NaN
3       B
4    3.14
5     inf
dtype: object

In [52]:
# Returns a numeric representation of the array, identifying unique values:
labels, uniques = pd.factorize(x)
labels

array([ 0,  0, -1,  1,  2,  3])

In [53]:
uniques

Index(['A', 'B', 3.14, inf], dtype='object')

In [54]:
pd.Categorical(x)

['A', 'A', NaN, 'B', 3.14, inf]
Categories (4, object): [3.14, inf, 'A', 'B']